In [1]:
import sqlite3
import pandas as pd

# Connect to SQLite database
conn = sqlite3.connect('../data/analytics.db')

In [14]:
# Export RFM Segments

rfm_query="""
SELECT 
    customer_id,
    MAX(ts) AS last_purchase,
    COUNT(DISTINCT transaction_id) AS frequency,
    SUM(total_amount) AS monetary,
    JulianDay((SELECT MAX(ts) FROM transactions)) - JulianDay(MAX(ts)) AS recency_days
FROM 
    transactions
GROUP BY 
    customer_id;
"""
rfm=pd.read_sql_query(rfm_query, conn)
rfm['recency_days'] = rfm['recency_days'].astype(int)

# Score Each RFM Component (1-5 scale)

# Higher frequency and monetary value are better
# Lower recency (more recent) is better
rfm['R_score']= pd.qcut(rfm['recency_days'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score']= pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score']= pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)

# Combine into RFM Score
rfm['RFM_score']= rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['RFM_score'] = rfm['RFM_score'].astype(int)

def segment(row):
    if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
        return "Loyal"
    elif row["R_score"] >= 4 and row["F_score"] <= 2:
        return "New"
    elif row["R_score"] <= 2 and row["F_score"] >= 3:
        return "At Risk"
    else:
        return "Occasional"

rfm["Segment"] = rfm.apply(segment, axis=1)

rfm.to_csv('../data/processed/rfm_segments.csv', index=False)
print("RFM segments exported to '../data/processed/rfm_segments.csv'")

RFM segments exported to '../data/processed/rfm_segments.csv'


In [11]:
# Export Product and Category performance
query="""
SELECT
    p.category,
    p.brand,
    p.name as product_name,
    SUM(ti.quantity * ti.price) AS revenue,
    SUM(ti.quantity) AS total_units
FROM
    transaction_items ti
JOIN
    products p ON ti.product_id = p.product_id
GROUP BY
    p.category, p.brand, p.product_id
ORDER BY
    revenue DESC;
"""
product_performance=pd.read_sql_query(query, conn)
product_performance.to_csv('../data/processed/product_performance.csv', index=False)
print("Product performance exported to '../data/processed/product_performance.csv'")

Product performance exported to '../data/processed/product_performance.csv'


In [6]:
# Export time-based sales data
time_query="""
SELECT
    STRFTIME('%Y-%m-%d', ts) AS date,
    STRFTIME('%w', ts) AS day_of_week,
    STRFTIME('%H', ts) AS hour,
    SUM(total_amount) AS daily_revenue
FROM
    transactions
GROUP BY
    date, day_of_week, hour
ORDER BY
    date, hour;
"""
time_sales=pd.read_sql_query(time_query, conn)
time_sales.to_csv('../data/processed/time_sales.csv', index=False)
print("Time-based sales data exported to '../data/processed/time_sales.csv'")

Time-based sales data exported to '../data/processed/time_sales.csv'
